# Walkthrough — Scalable DRL for the Non-Stationary SCLSP

Van Hezewijk, Dellaert, Van Jaarsveld (2025), *IJPE* 284:109601.

This notebook connects each paper section to the code in `src/` with **runnable sanity checks**.
Everything runs on CPU in seconds at toy scale. The mechanics are identical to the paper — only the
sizes (`N`, `M`, `H`) shrink. For quantitative reproduction see `REPRODUCTION_NOTES.md`.

Pipeline: **demand (DGP) → decomposed MDP env → AMBS rollout → MLP policy → DCL training → evaluation**.

## 1. Setup

We add `src/` to the path and load `configs/base.yaml`, then shrink the DCL/eval scale so the
notebook runs instantly. The paper scale (Table 2) needs a cluster.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join('..', 'src')))
import numpy as np

from utils import load_config, build_action_space, set_global_seed

config = load_config(os.path.join('..', 'configs', 'base.yaml'))
# Toy scale for the walkthrough (paper values stay documented in the yaml):
config['dcl'].update(dict(N_demo=40, M_demo=4, H_demo=5, generations=1,
                          epochs_per_generation=3, batch_size=16))
config['evaluation']['n_runs_demo'] = 30
set_global_seed(config['seed'])
print('K =', config['problem']['K'], '| demand mode =', config['demand']['mode'])

## 2. The demand process (DGP) — §1, §5.2

> "... an generative ARIMA type demand model, which enables the policy to learn to account for
> leadtime demand uncertainty associated with forecast inaccuracy and auto-correlated demand." — §2

**Background.** The whole method trains in simulation, so the demand model *is* the data. The exact
process is van Hezewijk et al. (2023a) — not in this paper — so `src/demand.py` implements an
IMA(0,1,1)-style **approximation** (discrete, non-negative, mean drifts with smoothing `α`). The
state carries the per-period forecast `(μ_t, σ_t)`, which is what lets one policy survive demand
changes without retraining (§4.3).

In [ ]:
from demand import build_demand_process

rng = np.random.default_rng(0)
dgp = build_demand_process(config)
f0 = dgp.reset(rng)
print('initial forecast mean:', f0.mu, '| std:', np.round(f0.sigma, 3))
demands = [dgp.step(rng)[0] for _ in range(5)]
print('5 periods of demand (product 0):', [int(d[0]) for d in demands])
assert (f0.mu >= 0).all() and all((d >= 0).all() for d in demands), 'demand must be non-negative'
print('\u2713 DGP produces non-negative integer demand with a forecast')

### What would happen if α = 0?
`α` is the *degree of non-stationarity* (§5.2.2). With `α=0` the level never drifts — demand is
stationary noise around `μ_0`. Larger `α` lets the mean random-walk further, which (Fig. 3–4)
raises the risk of capacity shortages.

In [ ]:
from demand import NonStationaryIMADemand
for a in (0.0, 0.05):
    d = NonStationaryIMADemand(K=1, mu0=2.0, cov0=1.0, alpha=a)
    r = np.random.default_rng(1); d.reset(r)
    levels = [float(d.step(r)[1].mu[0]) for _ in range(50)]
    print(f'alpha={a}: level range over 50 periods = [{min(levels):.2f}, {max(levels):.2f}]')

## 3. The decomposed MDP environment — §3 (the core contribution)

> "we propose to decompose the problem ... by decomposing the full period decision into
> sub-decisions." — §1
>
> "At time τ, first we take the 'first-stage' sub-decision of which product to produce ... Then we
> take the 'second-stage' sub-decision of how much of that product to produce." — §3.1

**State (Eq.1):** `s_τ = {I_{i,t}, μ_{i,t}, σ_{i,t}, ω_i, τ, q_{i,t}}`.
**Actions:** `A1 ∈ {p0,…,pK}` (Eq.2), `A2 ∈ {q1,…,q_{C−τ}}` (Eq.3).
One `env.step` = one sub-decision. A period closes on `p0` or capacity exhaustion; demand and
holding/backorder costs (Eq.4) land then.

In [ ]:
from env import SCLSPEnv, Stage

aspace = build_action_space(config)
print('unified action space size |A| =', aspace.size,
      '= (K+1 first-stage) +', aspace.n_second, 'restricted quantities (Eq.6/7)')

env = SCLSPEnv(config, aspace)
env.attach_demand(build_demand_process(config))
s = env.reset(np.random.default_rng(0))
print('capacity C = f_c * sum(mu_0) =', env.C)
print('start: stage =', Stage(s.stage).name, '| omega =', s.omega, '| tau =', s.tau)

# Action masking depends on the stage (§4.4):
mask = env.legal_action_mask()
assert mask[aspace.stop_index()], 'p0 (stop) must always be legal at first stage'
assert mask[1:aspace.n_first].any(), 'at least one product must be producible'
assert not mask[aspace.n_first:].any(), 'quantities must be masked OUT during first stage'
print('\u2713 first-stage mask exposes p0 + products, hides quantities')

In [ ]:
# Drive one full period by hand: set up product 0, then produce a quantity.
env.reset(np.random.default_rng(0))
_, r1, _, info1 = env.step(aspace.product_index(0), rng)   # first stage: choose product 0
print('after 1st-stage setup: reward =', r1, '(= -k_i setup cost)', '| info:', info1)
assert env.state.stage == Stage.SECOND, 'choosing a product must move us to the 2nd stage'
qmask = env.legal_action_mask()
assert qmask[aspace.n_first:].any() and not qmask[:aspace.n_first].any(), 'now only quantities legal'

q_slot = np.flatnonzero(qmask)[0]
_, r2, _, _ = env.step(int(q_slot), rng)                   # second stage: produce
print('after 2nd-stage produce: reward =', r2, '| tau used =', env.state.tau,
      '| inventory[0] =', env.state.I[0])
assert env.state.stage == Stage.FIRST, 'after producing we return to the first stage'
print('\u2713 two-stage sub-decision cycle works')

**Why the decomposition matters (numbers).** A full-period decision picks a quantity for every
product at once: with `Q` quantity levels and `K` products that is `Q^K` actions. The decomposed
action space is `|A1| + |A2*| = (K+1) + |grid|` — linear in `K`. That is the paper's mechanism for
scaling to more products (§4.4).

In [ ]:
K = config['problem']['K']; Q = aspace.n_second
print(f'full-period action space ~ Q^K = {Q}^{K} = {Q**K:,}')
print(f'decomposed action space  = (K+1)+Q = {aspace.size}')
assert aspace.size < Q**K, 'decomposition must shrink the action space'
print('\u2713 exponential \u2192 linear')

## 4. The AMBS rollout policy — §4.2, Algorithm 1

> "we perform a setup for the product i that has the largest expected gap to the reorder level ...
> For this product i, we will attempt to produce the quantity needed to reach the order up to
> level ... limited by the capacity available." — §4.2

It is both the **benchmark** and DCL's **generation-1 rollout policy**. (Order-up-to / continue lines
were reconstructed from a table image — see REPRODUCTION_NOTES.)

In [ ]:
from rollout import AMBSRolloutPolicy

ambs = AMBSRolloutPolicy(config, aspace)
env.reset(np.random.default_rng(0))
# Force a deep backorder on product 2 -> AMBS should prioritize setting it up.
env.state.I[:] = 5.0
env.state.I[2] = -10.0
a = ambs.act(env)
print('chosen first-stage action index:', a, '-> product', a - 1 if a > 0 else 'STOP')
assert a == aspace.product_index(2), 'largest gap (product 2) should be chosen'
print('\u2713 AMBS targets the product furthest below its reorder level')

## 5. The policy network — §4.3, §4.4

> "we adopt a dense (fully connected) neural network that takes as input a vector representation of
> the state ... and that has one output for every possible action." — §4.3
>
> "the representation for the product that is currently set up is presented first in the vector." — §4.3

The net outputs one logit per unified action; invalid logits are masked to −∞ (§4.4). It is trained
as a **classifier** of the best action (§4.1) — no value head, no policy gradient.

In [ ]:
import torch
from policy import build_policy_net, encode_state, state_dim, NeuralPolicy

env.reset(np.random.default_rng(0))
vec = encode_state(env.state, env)
print('state vector dim =', vec.shape[0], '= 3 global + K*7 =', state_dim(K))
assert vec.shape[0] == state_dim(K)

net = build_policy_net(config, aspace)
x = torch.from_numpy(vec).float().unsqueeze(0)              # (1, in_dim)
mask = torch.from_numpy(env.legal_action_mask()).unsqueeze(0)  # (1, |A|)
logits = net.masked_logits(x, mask)                        # (1, |A|)
assert logits.shape == (1, aspace.size)
neg_inf = torch.finfo(logits.dtype).min  # code uses finfo.min (not -inf) for softmax safety
assert (logits[0, ~mask[0].bool()] <= neg_inf).all(), 'masked-out actions must be driven to -inf'
print('✓ net maps state', tuple(x.shape), '-> masked logits', tuple(logits.shape))
print('  params:', sum(p.numel() for p in net.parameters()))

## 6. Deep Controlled Learning — §4.1

Per state: roll out every action `M` times for `H` periods (with **Common Random Numbers** and
**Sequential Halving**), label the cheapest action, train the net to classify it. Below we inspect a
single state's best-action estimate, then run one tiny generation end-to-end.

In [ ]:
from dcl import estimate_best_action
import copy

env.reset(np.random.default_rng(0))
snap = copy.deepcopy(env)
valid = [a for a, ok in enumerate(snap.legal_action_mask()) if ok]
best = estimate_best_action(snap, valid, ambs, M=8, H=5, crn_seed=123,
                            use_sequential_halving=True, use_crn=True)
print('valid actions at this state:', valid)
print('DCL best-action label (argmin estimated cost):', best)
assert best in valid, 'the labeled action must be a legal action'
print('\u2713 CRN + Sequential Halving return a legal best action')

In [ ]:
# One tiny DCL generation (mechanics only — not convergence).
from dcl import train_dcl
env2 = SCLSPEnv(config, aspace); env2.attach_demand(build_demand_process(config))
trained = train_dcl(config, env2, aspace, verbose=True)
for name, p in trained.named_parameters():
    assert torch.isfinite(p).all(), f'non-finite weights in {name}'
print('\u2713 DCL generation completed with finite weights')

## 7. Evaluation — §5.1

> "10,000 simulation runs with a length of 100 periods ... All policies are encountering the same
> demand sequences ... to enable a fair comparison." — §5.1

Metric: average cost per period; `Δ = (policy − AMBS)/AMBS`. **At this toy scale DCL will NOT beat
AMBS** — `N=40, M=4, 1 generation` is far below Table 2. This cell verifies the harness, not the
result. Set `dcl.use_paper_scale: true` on real hardware for the published Δ (Tables 3/5).

In [ ]:
from evaluate import evaluate
policies = {'AMBS': ambs, 'DCL': NeuralPolicy(trained, aspace)}
results = evaluate(config, policies, demo=True)
for name, (cost, delta) in results.items():
    tag = '' if name == 'AMBS' else f'   Delta_vs_AMBS = {delta:+.1%}'
    print(f'{name:6s}: avg cost/period = {cost:8.2f}{tag}')
assert results['AMBS'][0] > 0, 'AMBS should incur positive cost'
print('\u2713 evaluation harness runs (toy numbers are noisy by design)')

## 8. Common pitfalls

1. **PPO fails here (§4.1).** With one env-step per sub-decision, PPO learns to stall the period
   (7.4 actions/period, 3.6× worse than AMBS) because costs only land at period end. That is *why*
   the paper uses DCL — do not swap in vanilla policy-gradient.
2. **Eq.4 / Algorithm 1 are reconstructions.** The reward formula and the rollout's order-up-to /
   continue lines were recovered from table/equation images. Verify against the original PDF before
   trusting absolute numbers (REPRODUCTION_NOTES items 1, 3).
3. **The DGP is an approximation.** The exact van Hezewijk (2023a) process is not in this paper.
   Absolute costs depend heavily on it; swap it in for faithful reproduction (item 2).
4. **Two cost presets.** §5.1 stationary uses `k=200, θ=0`; Table 4 non-stationary uses `k=100,
   θ=1`. Mixing them silently corrupts comparisons.
5. **Action masking is not optional (§4.4).** The net shares one output head across both stages;
   forgetting to mask lets it pick a quantity during the first stage (or an illegal product),
   producing nonsense.
6. **Common Random Numbers (§4.1).** Evaluate all actions of a state against the *same* demand
   draws. Using independent draws per action buries the signal in variance — the failure mode DCL
   exists to fix.
7. **Demo vs paper scale.** `N/M/H` below Table 2 will not reproduce Δ. This is expected, not a bug.

## Further reading
- Temizöz et al. (2023) — *Deep Controlled Learning for inventory control* (the DCL method).
- van Hezewijk et al. (2023a) — the non-stationary demand process used as the DGP.
- van Hezewijk et al. (2023b) — the prior PPO / full-period AMBS formulation.
- Boute et al. (2021) — *Deep reinforcement learning for inventory control: a roadmap*.

See `PAPER_GUIDE.md` for a section-by-section narrative and `REPRODUCTION_NOTES.md` for every
flagged assumption.